# 🟦 Pattern 1 — Reactive Agents (LangGraph)

> **One-line definition:** the agent looks at the *current* input, decides once, acts, and stops.

---

## 1. Mental Model

```
Input  →  Decide  →  Act  →  Output
                (no loop, no memory, no plan)
```

- **Stateless** — each request is judged on its own.
- **No look-ahead** — it never asks "what happens after this step?"
- **No look-back** — it never asks "what did I learn last time?"
- The mapping is essentially a **function**: `state → action`.

---

## 2. Key Properties (pointwise)

| Property | Reactive agent |
|---|---|
| Planning | ❌ none |
| Loop | ❌ single pass |
| Memory | ❌ none |
| Reflection | ❌ none |
| Tools | ✅ optional (usually 0 or 1 call) |
| Latency | ⚡ lowest of all patterns |
| Cost | 💰 lowest (1 LLM call) |
| Predictability | ✅ highest |

---

## 3. When to use it

✅ **Good fit**
- Classification / routing / triage
- Translation, summarisation, extraction
- "Call exactly one API and format the answer"
- Any task where the *steps are already known* at design time

❌ **Bad fit**
- Task needs discovery ("find out X, then depending on X do Y")
- Task needs several dependent tool calls
- Task needs to recover from its own mistakes

---

## 4. The confusing part (resolved 👇)

**Q: "My reactive agent calls a tool. Doesn't that make it ReAct?"**

No. The difference is **who controls the loop**.

| | Reactive | ReAct |
|---|---|---|
| Tool call | Fires once, result is returned to user or formatted | Result is **fed back to the LLM**, which decides the *next* action |
| Graph shape | `START → node → END` (a line) | `START → agent ⇄ tools → END` (a cycle) |
| Number of LLM calls | Fixed (1–2) | Unbounded until the LLM stops |

👉 **A cycle in the graph is the dividing line.** No cycle = reactive.

**Q: "Isn't a reactive agent just a chain?"**

Essentially yes. Reactive is the *baseline*. It's included in the taxonomy because every other
pattern is defined as "reactive + something" (+ loop, + plan, + critique, + memory).

---

## 5. Graph we will build

```
        START
          │
          ▼
   ┌──────────────┐
   │  classify    │   1 LLM call, structured output
   └──────────────┘
          │
          ▼   (conditional edge — routing, NOT looping)
   ┌──────┬───────┬────────┐
   ▼      ▼       ▼        ▼
billing  tech   refund   general
   └──────┴───────┴────────┘
          │
          ▼
         END
```

⚠️ Note the branches **fan out and stop**. Nothing points backwards. That's what makes it reactive.


## 0. Setup

**Install once:**

```bash
pip install langgraph langchain-openai langchain-core
```

**Set your API key** (any chat model works — swap the import if you use Anthropic/Ollama).


In [ ]:
# --- Standard setup used by every notebook in this series ---
import os, getpass

def _set(var: str):
    """Prompt for a key only if it's not already in the environment."""
    if not os.environ.get(var):
        os.environ[var] = getpass.getpass(f"{var}: ")

_set("GROQ_API_KEY")

from langchain_groq import ChatGroq

# temperature=0 -> deterministic-ish output, easier to reason about while learning
llm = ChatGroq(model="llama-3.3-70b-versatile", temperature=0)
print("LLM ready")

---

## 6. Step 1 — Define the State

**Rules of thumb:**
- State is a `TypedDict` shared by every node.
- Each node returns a **partial dict** — LangGraph merges it in.
- Without a *reducer*, a returned key **overwrites** the old value (last write wins).


In [ ]:
from typing import TypedDict, Literal
from pydantic import BaseModel, Field


class ReactiveState(TypedDict):
    """Shared state. Every node reads it and returns a partial update."""
    query: str                                  # user's raw message (input)
    category: str                               # filled by the classify node
    response: str                               # filled by whichever handler runs


class Classification(BaseModel):
    """Pydantic schema -> forces the LLM to emit valid, parseable structured output.
    This is far more reliable than parsing free text with regex."""
    category: Literal["billing", "technical", "refund", "general"] = Field(
        description="The support category this query belongs to"
    )

---

## 7. Step 2 — The nodes

Every node has the same signature:

```python
def node(state: State) -> dict:   # returns a PARTIAL update
```


In [ ]:
def classify(state: ReactiveState) -> dict:
    """The ONLY LLM call in this graph. Pure input -> decision."""
    # with_structured_output binds the Pydantic schema to the model
    structured_llm = llm.with_structured_output(Classification)

    result = structured_llm.invoke(
        f"Classify this customer support query.\n\nQuery: {state['query']}"
    )
    print(f"  [classify] -> {result.category}")

    # Return ONLY the key we changed. LangGraph merges it into state.
    return {"category": result.category}


# --- Handlers: deterministic, no LLM. This is deliberate. ---
# A reactive agent should push as much as possible into cheap, testable code.

def handle_billing(state: ReactiveState) -> dict:
    return {"response": "Routed to Billing. Avg. response time: 4 hours."}


def handle_technical(state: ReactiveState) -> dict:
    return {"response": "Routed to Tech Support. Please share your error logs."}


def handle_refund(state: ReactiveState) -> dict:
    return {"response": "Refund request created. Processed within 5 business days."}


def handle_general(state: ReactiveState) -> dict:
    return {"response": "Thanks for reaching out! An agent will reply shortly."}

---

## 8. Step 3 — The router (conditional edge)

**Critical distinction:**

| | Node | Conditional edge function |
|---|---|---|
| Returns | a **state update** (`dict`) | a **node name** (`str`) |
| Changes state? | ✅ yes | ❌ no — it only *reads* |
| Purpose | do work | pick the next node |

A common beginner bug is trying to update state from a router. It silently does nothing.


In [ ]:
def route(state: ReactiveState) -> str:
    """Reads state, returns the NAME of the next node. Never mutates state."""
    return {
        "billing":   "handle_billing",
        "technical": "handle_technical",
        "refund":    "handle_refund",
        "general":   "handle_general",
    }[state["category"]]

---

## 9. Step 4 — Build & compile the graph

**The v1.0 API (use this, not the old tutorials):**

| ✅ Current | ❌ Deprecated |
|---|---|
| `builder.add_edge(START, "node")` | `builder.set_entry_point("node")` |
| `builder.add_edge("node", END)` | `builder.set_finish_point("node")` |


In [ ]:
from langgraph.graph import StateGraph, START, END

builder = StateGraph(ReactiveState)

# 1. Register nodes: add_node(name, function)
builder.add_node("classify", classify)
builder.add_node("handle_billing", handle_billing)
builder.add_node("handle_technical", handle_technical)
builder.add_node("handle_refund", handle_refund)
builder.add_node("handle_general", handle_general)

# 2. Entry point
builder.add_edge(START, "classify")

# 3. The fan-out. Signature: (source_node, router_fn, path_map)
#    path_map tells LangGraph which node names the router can possibly return.
#    It is optional but STRONGLY recommended -> enables correct graph drawing.
builder.add_conditional_edges(
    "classify",
    route,
    ["handle_billing", "handle_technical", "handle_refund", "handle_general"],
)

# 4. Every handler terminates. No back-edges == reactive.
for handler in ["handle_billing", "handle_technical", "handle_refund", "handle_general"]:
    builder.add_edge(handler, END)

graph = builder.compile()   # compile() validates the graph and returns a Runnable
print("Graph compiled")

### Visualise it

If this cell fails, it's only a rendering dependency — the graph itself is fine.


In [ ]:
from IPython.display import Image, display

try:
    display(Image(graph.get_graph().draw_mermaid_png()))
except Exception as e:
    # Fallback: ASCII / mermaid text always works offline
    print(graph.get_graph().draw_mermaid())

---

## 10. Step 5 — Run it


In [ ]:
queries = [
    "I was charged twice this month!",
    "The app crashes when I open settings",
    "I want my money back for last month's plan",
    "What are your office hours?",
]

for q in queries:
    print(f"\nQ: {q}")
    result = graph.invoke({"query": q})     # invoke = run to completion
    print(f"  -> [{result['category']}] {result['response']}")

### Streaming (see each node fire)

`stream_mode` options worth knowing:

| Mode | Yields |
|---|---|
| `"updates"` | only what each node **returned** (best for debugging) |
| `"values"` | the **full state** after each step |
| `"messages"` | LLM tokens as they're generated |


In [ ]:
for chunk in graph.stream(
    {"query": "My invoice looks wrong"},
    stream_mode="updates",
):
    print(chunk)

---

## 11. Variant — Reactive agent with a single tool

Still reactive: **one** tool call, result formatted, done. No loop back to the LLM.


In [ ]:
from langchain_core.tools import tool


@tool
def get_order_status(order_id: str) -> str:
    """Look up the delivery status of an order by its ID."""
    # Fake DB. In production this is your real service call.
    return {"A1": "Shipped, arrives Friday", "B2": "Processing"}.get(order_id, "Not found")


class ToolState(TypedDict):
    query: str
    response: str


def reactive_tool_node(state: ToolState) -> dict:
    """Decide -> act -> format. Exactly one pass."""
    model = llm.bind_tools([get_order_status])
    ai_msg = model.invoke(state["query"])

    if not ai_msg.tool_calls:
        return {"response": ai_msg.content}     # no tool needed, answer directly

    # Execute the FIRST tool call only, then stop.
    # (A ReAct agent would instead send the result back to the LLM here.)
    call = ai_msg.tool_calls[0]
    tool_result = get_order_status.invoke(call["args"])
    return {"response": f"Order {call['args']['order_id']}: {tool_result}"}


tool_builder = StateGraph(ToolState)
tool_builder.add_node("act", reactive_tool_node)
tool_builder.add_edge(START, "act")
tool_builder.add_edge("act", END)
tool_graph = tool_builder.compile()

print(tool_graph.invoke({"query": "Where is order A1?"})["response"])

---

## 12. Cheat Sheet

```
DEFINITION   state -> action, one pass, no cycle
GRAPH SHAPE  a line or a fan-out tree (DAG). Never a cycle.
LLM CALLS    fixed and known in advance
STATE        write-only, discarded after the run
COST         lowest
USE WHEN     the steps are known at design time
AVOID WHEN   the agent must discover the path
```

**API essentials**

| Task | Code |
|---|---|
| Define state | `class S(TypedDict): ...` |
| Add node | `builder.add_node("name", fn)` |
| Fixed edge | `builder.add_edge("a", "b")` |
| Branching | `builder.add_conditional_edges("a", router, ["b","c"])` |
| Entry / exit | `add_edge(START, "a")` / `add_edge("a", END)` |
| Compile | `graph = builder.compile()` |
| Run | `graph.invoke({...})` |
| Structured output | `llm.with_structured_output(PydanticModel)` |

---

## 13. Decision Tree

```
Do I know all the steps before running?
├── YES
│   └── Does it need >2 dependent LLM calls?
│       ├── NO  ──────────────► ✅ REACTIVE  (this notebook)
│       └── YES ──────────────► → Planning agent (Pattern 3)
└── NO
    └── Must it discover the path via tool results?
        ├── YES ──────────────► → ReAct agent (Pattern 2)
        └── Must it critique itself?
            └── YES ──────────► → Reflective agent (Pattern 4)
```

---

## 14. Next

➡️ **Pattern 2 — Reasoning (ReAct) Agents**: we add the cycle back to the LLM, and the
agent starts discovering its own path.
